# Phase 3 演習 — 材料力学の再導出

対応する本文：[`docs/texts/phase3-member-mechanics.md`](../docs/texts/phase3-member-mechanics.md)

このNotebookは提出用です。各 **あなたの回答** を埋め、`TODO` のあるコードを完成させて実行してください。記号、単位、採用した仮定を明示し、数値だけでなくCAEとの差が何を意味するかも説明してください。

## 提出情報

- 氏名・日付：
- 実行環境（任意）：
- 参照した資料・CAE結果（任意）：

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

---
## 演習 1 — 断面応力から軸力と曲げモーメントを戻す

幅 $b=20\ \mathrm{mm}$、高さ $h=40\ \mathrm{mm}$ の長方形断面で、$y=0$ を図心、$y>0$ を上側とします。軸応力が

$$\sigma_{xx}(y)=\frac{N}{A}-\frac{M_z}{I_z}y$$

で表され、$N=24\ \mathrm{kN}$、$M_z=1.6\times10^5\ \mathrm{N\,mm}$ とします。

1. $A$ と $I_z$ を求め、型・役割・単位を説明する。
2. 上縁、図心、下縁の $\sigma_{xx}$ を求める。引張・圧縮も示す。
3. 数値積分で $\int_A\sigma_{xx}dA=N$、$-\int_A y\sigma_{xx}dA=M_z$ を確認する。
4. 節点平均応力からこの積分を行う際の注意を一つ述べる。

### あなたの回答（演習 1-1, 1-2, 1-4）

1. 
2. 
4. 

In [ ]:
# 演習 1-3：長方形断面をy方向の短冊へ分割して断面積分する。
b_mm = 20.0
h_mm = 40.0
N_N = 24.0e3
Mz_Nmm = 1.6e5
n_strip = 20001

# TODO: 面積 A_mm2 と断面二次モーメント Iz_mm4 を実装する。
A_mm2 = ...
Iz_mm4 = ...

y_mm = np.linspace(-h_mm / 2.0, h_mm / 2.0, n_strip)
# TODO: sigma_xx_mpa(y) を実装する。N/mm^2 = MPa。
sigma_xx_mpa = ...

# TODO: np.trapezoid（古いNumPyではnp.trapz）を使い、幅bを掛けてNとMzを戻す。
N_recovered_N = ...
Mz_recovered_Nmm = ...

print('A [mm^2] =', A_mm2)
print('Iz [mm^4] =', Iz_mm4)
print('sigma(bottom, center, top) [MPa] =', sigma_xx_mpa[[0, n_strip // 2, -1]])
print('recovered N [N] =', N_recovered_N)
print('recovered Mz [N mm] =', Mz_recovered_Nmm)
assert np.isclose(N_recovered_N, N_N, rtol=1e-6)
assert np.isclose(Mz_recovered_Nmm, Mz_Nmm, rtol=1e-6)

---
## 演習 2 — 軸剛性：段付き棒を積分する

鋼製の段付き棒に引張力 $N=30\ \mathrm{kN}$ が働きます。区間1は $L_1=300\ \mathrm{mm}, A_1=200\ \mathrm{mm^2}$、区間2は $L_2=200\ \mathrm{mm}, A_2=100\ \mathrm{mm^2}$、$E=210\ \mathrm{GPa}$ です。

1. 各区間の応力、ひずみ、伸びと全伸びを求める。
2. なぜ伸びは $N/A$ の平均からではなく $N/(EA)$ の区間積分で足し合わせるのか説明する。
3. 段差の角に生じるソリッドCAEの最大応力が $N/A_2$ より高い理由と、比較に適した位置を述べる。

### あなたの回答（演習 2）

1. 
2. 
3. 

In [ ]:
# 演習 2：一般の区分一定棒にも使える関数を完成させる。
def axial_response(N_N, lengths_mm, areas_mm2, E_mpa):
    lengths_mm = np.asarray(lengths_mm, dtype=float)
    areas_mm2 = np.asarray(areas_mm2, dtype=float)
    if lengths_mm.shape != areas_mm2.shape or np.any(lengths_mm <= 0) or np.any(areas_mm2 <= 0):
        raise ValueError('lengthsとareasは同じ形の正値配列とする')
    # TODO: 各区間の応力、ひずみ、伸びを実装する。
    sigma_mpa = ...
    strain = ...
    elongation_mm = ...
    return sigma_mpa, strain, elongation_mm

sigma, strain, elongation = axial_response(30e3, [300.0, 200.0], [200.0, 100.0], 210e3)
print('sigma [MPa] =', sigma)
print('strain [-] =', strain)
print('elongation [mm] =', elongation)
print('total elongation [mm] =', elongation.sum())
assert np.allclose(sigma, [150.0, 300.0])
assert np.isclose(elongation.sum(), 0.5)

---
## 演習 3 — 曲げ：応力分布とたわみを同じ $EI$ から得る

長さ $L=800\ \mathrm{mm}$ の片持ち梁の自由端に $P=500\ \mathrm{N}$ が作用します。断面は幅 $b=30\ \mathrm{mm}$、高さ $h=60\ \mathrm{mm}$、$E=70\ \mathrm{GPa}$ です。

1. 固定端の最大曲げ応力と自由端たわみの大きさを求める。
2. $h$ だけを2倍にしたとき、最大応力と自由端たわみが何倍になるか、$I_z$ の依存性から説明する。
3. 同じモデルをビーム要素とソリッド要素で解いたとき、固定端ピーク応力ではなく比較すべき量を二つ挙げる。
4. 梁が短く厚くなると、Euler–Bernoulli式がたわみを過小評価し得る理由を述べる。

### あなたの回答（演習 3）

1. 
2. 
3. 
4. 

In [ ]:
# 演習 3：断面高さを変えたパラメトリック計算を完成させる。
def cantilever_tip_load(P_N, L_mm, b_mm, h_mm, E_mpa):
    # TODO: Iz、固定端外縁応力の絶対値、自由端たわみの絶対値を実装する。
    Iz_mm4 = ...
    sigma_max_mpa = ...
    tip_deflection_mm = ...
    return Iz_mm4, sigma_max_mpa, tip_deflection_mm

base = cantilever_tip_load(500.0, 800.0, 30.0, 60.0, 70e3)
double_h = cantilever_tip_load(500.0, 800.0, 30.0, 120.0, 70e3)
print('base (Iz, sigma_max, tip_deflection) =', base)
print('double h =', double_h)
print('ratios (double/base) =', np.asarray(double_h) / np.asarray(base))
assert np.allclose(np.asarray(double_h) / np.asarray(base), [8.0, 0.25, 0.125])

---
## 演習 4 — ねじり：円形断面と非円形断面を区別する

長さ $L=1000\ \mathrm{mm}$、直径 $d=30\ \mathrm{mm}$ の中実円形軸に $T=250\ \mathrm{N\,m}$ を加えます。$G=80\ \mathrm{GPa}$ とします。

1. 極断面二次モーメント $J$、外周最大せん断応力、全ねじれ角を求める。
2. 応力が中心でゼロ、外周で最大になることを、運動学と構成則から説明する。
3. 同じ面積の長方形断面へ $J=\int_A\rho^2dA$ と $\tau=T\rho/J$ をそのまま使えない理由を述べる。
4. ソリッドCAEで円形軸の式を検証する際、トルクを加える端面から離れて評価する理由を述べる。

### あなたの回答（演習 4）

1. 
2. 
3. 
4. 

In [ ]:
# 演習 4：円形軸の半径方向分布を実装する。単位系はN, mm, MPa。
T_Nmm = 250.0e3
L_mm = 1000.0
d_mm = 30.0
G_mpa = 80.0e3
rho_mm = np.linspace(0.0, d_mm / 2.0, 101)

# TODO: J、せん断応力分布、ねじれ角を実装する。
J_mm4 = ...
tau_mpa = ...
twist_rad = ...

print('J [mm^4] =', J_mm4)
print('tau_max [MPa] =', tau_mpa[-1])
print('twist [rad, deg] =', twist_rad, np.rad2deg(twist_rad))
assert np.isclose(tau_mpa[0], 0.0)
assert np.all(np.diff(tau_mpa) > 0.0)

---
## 演習 5 — 座屈：固有値を耐荷力と読み違えない

長さ $L=1500\ \mathrm{mm}$、長方形断面 $b=20\ \mathrm{mm}, h=50\ \mathrm{mm}$ の鋼柱を考えます。$E=210\ \mathrm{GPa}$、降伏応力 $\sigma_y=300\ \mathrm{MPa}$ とします。

1. 二つの図心軸まわりの $I$ を求め、どちら向きに座屈しやすいか示す。
2. ピン–ピン（$K=1$）と固定–固定（理想値 $K=0.5$）のEuler荷重を求める。
3. ピン–ピンの場合の $P_{\mathrm{cr}}/A$ と $\sigma_y$ を比較し、弾性Euler座屈という仮定の自己整合性を論じる。
4. 線形固有値解析の1次モード最大値が 1.0 mm と表示された。この値を実変位とみなせない理由を説明する。
5. 実耐荷力へ近づける非線形解析で追加すべきものを三つ挙げる。

### あなたの回答（演習 5）

1. 
2. 
3. 
4. 
5. 

In [ ]:
# 演習 5：任意のKと二つの断面二次モーメントを扱う関数を完成させる。
def euler_buckling(E_mpa, area_mm2, inertias_mm4, L_mm, K):
    inertias_mm4 = np.asarray(inertias_mm4, dtype=float)
    # TODO: 断面二次半径、細長比、Euler荷重、臨界平均応力を実装する。
    radii_mm = ...
    slenderness = ...
    Pcr_N = ...
    sigma_cr_mpa = ...
    return radii_mm, slenderness, Pcr_N, sigma_cr_mpa

b_mm, h_mm = 20.0, 50.0
area_mm2 = b_mm * h_mm
I_about_z_mm4 = b_mm * h_mm**3 / 12.0
I_about_y_mm4 = h_mm * b_mm**3 / 12.0
for K in (1.0, 0.5):
    result = euler_buckling(210e3, area_mm2, [I_about_z_mm4, I_about_y_mm4], 1500.0, K)
    print(f'K={K}: radii, slenderness, Pcr[N], sigma_cr[MPa] =')
    for item in result:
        print(item)
    assert np.all(result[2] > 0.0)

---
## 演習 6 — CAE結果レビュー：差を仮定へ戻す

手元のCAE結果、または軸・曲げ・ねじり・座屈のいずれかを想定した解析について記入してください。機密値は無次元化・抽象化して構いません。

- 対象現象と採用した部材理論：
- 比較したCAE量と評価位置（積分点、断面力、節点変位、固有値など）：
- 手計算とCAEで一致した量、および差率：
- 外力・反力・断面力の釣合い確認：
- 部材理論の仮定のうち、モデルで満たすもの／満たさないもの：
- 差の原因候補を「物理モデル」「離散化」「出力処理」に分類：
- 次に行う一つの検証と、それで棄却したい原因候補：

### あなたの回答（演習 6）

- 対象現象と採用した部材理論：
- 比較したCAE量と評価位置：
- 一致した量と差率：
- 釣合い確認：
- 満たす仮定／満たさない仮定：
- 原因候補の分類：
- 次の検証：

## 振り返り

1. 四つの公式のうち、仮定からのつながりが最も明確になったもの：
2. 部材理論とCAEの差を、誤差以外の言葉で説明できた例：
3. まだ説明できない導出・適用限界：
4. 穴埋め実装は式と計算の接続に役立ったか。改善したい点：

この内容は `docs/learning-log.md` のPhase 3項目へ追記してください。